# Referencia rápida: Modelos de HuggingFace con PyTorch

> Tenla abierta mientras trabajas en los TODOs del workshop. Cada sección es ejecutable — corre las celdas para ver los outputs en vivo.

| Sección | Cuándo consultarla |
|---------|-------------------|
| 1. Cargar modelo | TODO 1, TODO 2 — primera celda del notebook |
| 2. Dispositivo | Error `tensors on same device` |
| 3. `model.eval()` | Resultados inconsistentes entre llamadas |
| 4. `no_grad()` | Memoria que crece, inferencia lenta |
| 5. Mover inputs | Error de device al llamar al modelo |
| 6. Extraer features | TODO 1, TODO 2 — implementar `get_*_embeddings` |
| 7. Volver a CPU | Error al llamar `.numpy()` |
| 8. Patrón completo | Antes de empezar TODO 1 o TODO 2 |
| Errores comunes | Cuando algo no funciona |

In [ ]:
%%capture
!pip install transformers torch Pillow

---
## 1. Cargar un modelo preentrenado

`from_pretrained` descarga los pesos y la configuración desde HuggingFace Hub.  
El **processor** agrupa el tokenizador de texto y el preprocesador de imagen en un solo objeto.

In [ ]:
from transformers import CLIPModel, CLIPProcessor

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

print(type(model))      # transformers.models.clip.modeling_clip.CLIPModel
print(type(processor))  # transformers.models.clip.processing_clip.CLIPProcessor

---
## 2. Dispositivo: CPU vs GPU

PyTorch **no mueve nada automáticamente** — tienes que hacerlo explícitamente.

> **Regla:** el modelo y los tensores de entrada deben estar **en el mismo dispositivo**.  
> Si el modelo está en `cuda` y los inputs en `cpu`, obtendrás un error.

In [ ]:
import torch

# Detectar qué hay disponible
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}")

# Mover el modelo al dispositivo (solo se hace una vez, al inicio)
model = model.to(device)
print(f"Modelo en: {next(model.parameters()).device}")

---
## 3. Modo evaluación vs entrenamiento

Los modelos de HuggingFace arrancan en **modo entrenamiento** por defecto.  
Para inferencia siempre cambia a modo evaluación:

| Modo | `model.train()` | `model.eval()` |
|------|----------------|----------------|
| Dropout | activo (aleatorio) | desactivado |
| BatchNorm | usa estadísticas del batch | usa estadísticas guardadas |
| Uso | entrenamiento | inferencia / evaluación |

In [ ]:
# Estado antes
print(f"¿En modo training? {model.training}")  # True

model.eval()

# Estado después
print(f"¿En modo training? {model.training}")  # False

---
## 4. `torch.no_grad()` — no calcular gradientes

Durante inferencia no necesitas gradientes. Desactivarlos **ahorra memoria y acelera el cómputo**.

> **¿Por qué importa?** PyTorch guarda el grafo computacional para el backward pass.  
> Si no vas a llamar `.backward()`, ese grafo solo ocupa memoria. `no_grad()` lo evita.

In [ ]:
# Sin no_grad: el tensor guarda el grafo → requires_grad=True
inputs = processor(text=["a dog"], return_tensors="pt")
inputs = {k: v.to(device) for k, v in inputs.items()}

features_with_grad = model.get_text_features(**inputs)
if not isinstance(features_with_grad, torch.Tensor):
    features_with_grad = features_with_grad.pooler_output
print(f"Con grad:    requires_grad={features_with_grad.requires_grad}")

# Con no_grad: más eficiente, sin grafo
with torch.no_grad():
    features_no_grad = model.get_text_features(**inputs)
    if not isinstance(features_no_grad, torch.Tensor):
        features_no_grad = features_no_grad.pooler_output
print(f"Sin grad:    requires_grad={features_no_grad.requires_grad}")

---
## 5. Mover tensores al dispositivo

El processor devuelve tensores **siempre en CPU**. Hay que moverlos al mismo dispositivo que el modelo.

In [ ]:
inputs = processor(text=["a dog"], return_tensors="pt")
print(f"Device de inputs antes: {inputs['input_ids'].device}")

# Opción A — iterar el dict (compatible con todas las versiones de transformers)
inputs_a = {k: v.to(device) for k, v in inputs.items()}
print(f"Device de inputs (opción A): {inputs_a['input_ids'].device}")

# Opción B — encadenar .to(device) directamente al processor
inputs_b = processor(text=["a dog"], return_tensors="pt").to(device)
print(f"Device de inputs (opción B): {inputs_b['input_ids'].device}")

# Ambas son equivalentes — en el workshop usamos la Opción A

---
## 6. Extraer features con CLIP

CLIP tiene dos métodos para extraer embeddings. Los embeddings **no están normalizados** al salir del modelo — hay que normalizarlos antes de calcular similitud coseno.

In [ ]:
import torch.nn.functional as F
from PIL import Image
import requests
from io import BytesIO

# ── Embedding de texto ────────────────────────────────────────────────────────
text_inputs = processor(
    text=["a dog", "a cat"],
    return_tensors="pt", padding=True, truncation=True, max_length=77
)
text_inputs = {k: v.to(device) for k, v in text_inputs.items()}

with torch.no_grad():
    text_features = model.get_text_features(**text_inputs)
    if not isinstance(text_features, torch.Tensor):
        text_features = text_features.pooler_output

print(f"Texto — shape antes de normalizar: {text_features.shape}")
print(f"Texto — norma antes:               {text_features.norm(dim=-1)}")

text_features = F.normalize(text_features, p=2, dim=-1)
print(f"Texto — norma después:             {text_features.norm(dim=-1)}")

In [ ]:
# ── Embedding de imagen ───────────────────────────────────────────────────────
# Imagen de ejemplo (descargada de internet)
url = "https://upload.wikimedia.org/wikipedia/commons/thumb/2/26/YellowLabradorLooking_new.jpg/320px-YellowLabradorLooking_new.jpg"
response = requests.get(url)
image = Image.open(BytesIO(response.content)).convert("RGB")

image_inputs = processor(images=[image], return_tensors="pt")
image_inputs = {k: v.to(device) for k, v in image_inputs.items()}

with torch.no_grad():
    image_features = model.get_image_features(**image_inputs)
    if not isinstance(image_features, torch.Tensor):
        image_features = image_features.pooler_output

image_features = F.normalize(image_features, p=2, dim=-1)

print(f"Imagen — shape: {image_features.shape}")
print(f"Imagen — norma: {image_features.norm(dim=-1)}")

# Similitud entre la imagen y los dos textos
scores = (image_features @ text_features.T).squeeze()
print(f"\nSimilitud imagen vs 'a dog': {scores[0]:.4f}")
print(f"Similitud imagen vs 'a cat': {scores[1]:.4f}")
print("→ La imagen de un perro debería tener mayor similitud con 'a dog'")

---
## 7. Mover resultados de vuelta a CPU

NumPy y Matplotlib solo trabajan con tensores en CPU. Después de la inferencia, mueve el resultado.

In [ ]:
# Los features están en el mismo device que el modelo
print(f"Device del embedding: {image_features.device}")

# Mover a CPU para usar con numpy / matplotlib
embedding_cpu = image_features.cpu()
print(f"Device después de .cpu(): {embedding_cpu.device}")

# Convertir a numpy
embedding_np = embedding_cpu.numpy()
print(f"Shape como numpy array: {embedding_np.shape}")

# ── Error común: llamar .numpy() en GPU ──────────────────────────────────────
# embedding_gpu.numpy()   ← RuntimeError: numpy() on GPU tensor
# Solución: .cpu().numpy()  o  .detach().cpu().numpy()

---
## 8. Patrón completo — TODO 1 y TODO 2

Juntando todo lo anterior, así se ve una función de embedding correcta.  
Este es exactamente el patrón que necesitas para implementar los TODOs 1 y 2.

In [ ]:
def get_embeddings_example(inputs_raw, model, processor, device):
    """
    Patrón completo para extraer embeddings normalizados.
    Sustituye inputs_raw por texts o images según el TODO.
    """
    # 1. Preprocesar y mover al dispositivo
    #    Para texto:   processor(text=inputs_raw, return_tensors="pt", padding=True, truncation=True, max_length=77)
    #    Para imagen:  processor(images=inputs_raw, return_tensors="pt")
    inputs = processor(text=inputs_raw, return_tensors="pt",
                       padding=True, truncation=True, max_length=77)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # 2. Inferencia sin gradientes
    with torch.no_grad():
        features = model.get_text_features(**inputs)  # o get_image_features
        if not isinstance(features, torch.Tensor):
            features = features.pooler_output

    # 3. Normalizar L2
    features = F.normalize(features, p=2, dim=-1)

    # 4. Devolver en CPU
    return features.cpu()


# Verificación
embs = get_embeddings_example(["a dog", "a cat", "a rocket"], model, processor, device)
assert embs.shape == (3, 512)
assert torch.allclose(embs.norm(dim=-1), torch.ones(3), atol=1e-5)
print(f"Shape: {embs.shape}  ✅")
print(f"Normas: {embs.norm(dim=-1)}  ✅")

---
## Errores comunes

| Error | Causa | Solución |
|-------|-------|----------|
| `Expected all tensors to be on the same device` | inputs en CPU, modelo en GPU | `{k: v.to(device) for k, v in inputs.items()}` |
| `Can't call numpy() on Tensor that requires grad` | tensor con gradiente | `.detach().cpu().numpy()` |
| `object has no attribute 'pooler_output'` | llamaste a `model.text_model()` en lugar de `model.get_text_features()` | usar `model.get_text_features(**inputs)` |
| Resultados distintos entre llamadas | modelo en modo train | llamar `model.eval()` al cargar |
| Consumo de memoria que crece | inferencia sin `no_grad()` | envolver en `with torch.no_grad():` |

In [ ]:
# ── Reproducir y solucionar el error más común ────────────────────────────────

inputs = processor(text=["a dog"], return_tensors="pt")
# inputs está en CPU, modelo en device

# ❌ Error si model está en GPU
# model.get_text_features(**inputs)   # RuntimeError: tensors on different devices

# ✅ Solución
inputs = {k: v.to(device) for k, v in inputs.items()}
with torch.no_grad():
    features = model.get_text_features(**inputs)
    if not isinstance(features, torch.Tensor):
        features = features.pooler_output
features = F.normalize(features, p=2, dim=-1).cpu()
print(f"✅ Sin errores | shape: {features.shape} | norma: {features.norm(dim=-1).item():.6f}")